# Comparative Analysis of ML Algorithms for SMS Spam Detection
**Academic Research Project | Subject: Research Methodologies (RM) — 7th Semester B.Tech CSE**
**Author:** Priyansu Das | **Institution:** Siksha 'O' Anusandhan University (ITER) | **Repository:** `priyansudas07/sms-spam-detection`

### Project Overview
This notebook presents an end-to-end Machine Learning research investigation for automated SMS spam detection using the official UCI SMS corpus (5,574 messages).
We implement a **$2 \times 4$ full factorial experiment** evaluating two feature extraction techniques:
1. **TF-IDF Vectorizer** (Sublinear term frequency inversely scaled by document rarity)
2. **CountVectorizer** (Discrete Bag-of-Words term frequency counts)

Across four machine learning classifiers:
1. **Multinomial Naive Bayes (MNB)**
2. **Bernoulli Naive Bayes (BNB)** *(Strict binary occurrence modeling with binarized features)*
3. **Logistic Regression (LR)**
4. **Linear Support Vector Machine (Linear SVM)** *(Calibrated probability outputs via Platt scaling)*

Total: **8 Experimental Combinations**.

### Methodological Rigor & Guarantees:
- **Zero Data Leakage:** All vectorization and text transformations are encapsulated inside `sklearn.pipeline.Pipeline` objects, fitted strictly on training folds.
- **Natural Class Distribution:** Preserves authentic class balance (87.56% Ham, 12.44% Spam) without artificial synthetic resampling.
- **Validation-Driven Model Selection:** Model selection is conducted strictly on training data via **Stratified 5-Fold Cross-Validation** using **Mean CV $F_1$-score**. The test partition ($N = 1,032$) remains untouched until final validation.
- **Benchmarking:** Rigorously compared against the peer-reviewed IEEE TENCON 2023 study (*Robert G. de Luna et al.*).


## 1. Environment Initialization & Libraries


In [ ]:
# Dependency installation (Uncomment for Google Colab execution)
# !pip install scikit-learn pandas numpy matplotlib seaborn nltk

import os
import re
import string
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.stem.porter import PorterStemmer

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Download essential NLTK stopwords if available, with robust fallback
try:
    nltk.download('stopwords', quiet=True)
    from nltk.corpus import stopwords
    _raw_stopwords = set(stopwords.words('english'))
except Exception:
    _raw_stopwords = set(ENGLISH_STOP_WORDS)

# Plotting configuration
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams['figure.figsize'] = (10, 5)
print("Environment successfully initialized.")


## 2. Dataset Acquisition & Inspection
We fetch the official **UCI SMS Spam Collection dataset** (ID: 228) and inspect raw data distributions.


In [ ]:
DATASET_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
data_dir = "data"
raw_file = os.path.join(data_dir, "SMSSpamCollection")
zip_file = os.path.join(data_dir, "smsspamcollection.zip")

os.makedirs(data_dir, exist_ok=True)

if not os.path.exists(raw_file):
    print("Downloading dataset from UCI repository...")
    headers = {'User-Agent': 'Mozilla/5.0'}
    req = urllib.request.Request(DATASET_URL, headers=headers)
    with urllib.request.urlopen(req) as resp, open(zip_file, 'wb') as f:
        f.write(resp.read())
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    if os.path.exists(zip_file):
        os.remove(zip_file)
    print("Dataset extracted successfully.")
else:
    print("Dataset already exists locally.")

df_raw = pd.read_csv(
    raw_file,
    sep='\t',
    names=['label', 'message'],
    header=None,
    encoding='utf-8',
    quoting=3
)

print(f"Total Raw Messages: {len(df_raw)}")
print(f"Missing Values:\n{df_raw.isnull().sum()}\n")
print("First 5 Sample Records:")
df_raw.head()


## 3. Data Cleaning & Natural Distribution Preservation
We perform string stripping, verify absence of conflicting duplicate labels, remove 414 duplicate messages, and preserve the authentic 87.56% : 12.44% natural class distribution.


In [ ]:
initial_count = len(df_raw)

# Label normalization and message whitespace stripping
df_clean = df_raw.dropna(subset=['label', 'message']).copy()
df_clean['label'] = df_clean['label'].astype(str).str.strip().str.lower()
df_clean['message'] = df_clean['message'].astype(str).str.strip()

# Check for conflicting duplicate messages (same text, different label)
msg_label_counts = df_clean.groupby('message')['label'].nunique()
conflicting = msg_label_counts[msg_label_counts > 1]
print(f"Conflicting Duplicate Messages Detected: {len(conflicting)}")

# Deduplication
df_clean = df_clean.drop_duplicates(subset=['message']).reset_index(drop=True)
df_clean['label_num'] = df_clean['label'].map({'ham': 0, 'spam': 1})

print(f"Cleaned Dataset Total: {len(df_clean)} unique messages (Dropped {initial_count - len(df_clean)} duplicates)")
print("\nClass Distribution:")
class_summary = pd.DataFrame({
    'Count': df_clean['label'].value_counts(),
    'Percentage (%)': (df_clean['label'].value_counts(normalize=True) * 100).round(2)
})
display(class_summary)


## 4. Methodological NLP Preprocessing
To prevent destruction of critical smishing indicators while reducing vocabulary sparsity:
1. **URLs replaced with atomic token `urltoken`** (URLs are high-confidence spam markers).
2. **Emails replaced with `emailtoken`**.
3. **Currency symbols (`$`, `£`, `€`, `₹`) mapped to `currencytoken`**.
4. **Meaningful negation words preserved** (`not`, `no`, `never`, `cannot`, `wont`, `dont`, etc.) to prevent polarity inversion.
5. **Porter stemming** applied to normalize word roots.


In [ ]:
# Negation preservation set
NEGATION_WORDS = {
    "not", "no", "nor", "neither", "never", "cannot", "cant", "can't",
    "wont", "won't", "dont", "don't", "isnt", "isn't", "arent", "aren't",
    "wasnt", "wasn't", "werent", "weren't", "without", "against"
}

ACTIVE_STOPWORDS = _raw_stopwords - NEGATION_WORDS
STEMMER = PorterStemmer()

def preprocess_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    
    # Preserve domain spam indicators as atomic semantic tokens
    text = re.sub(r'https?://\S+|www\.\S+', ' urltoken ', text)
    text = re.sub(r'\S+@\S+', ' emailtoken ', text)
    text = re.sub(r'[$£€₹]', ' currencytoken ', text)
    
    # Strip punctuation and non-alphanumeric characters
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    tokens = text.split()
    stemmed = [
        STEMMER.stem(w) for w in tokens
        if (w not in ACTIVE_STOPWORDS or w in {'urltoken', 'emailtoken', 'currencytoken'}) and len(w) > 1
    ]
    return " ".join(stemmed)

# Demonstration
sample_msg = "WINNER!! As a valued customer, you have won £900 prize! Claim now at http://win.it"
print("Original Raw SMS :", sample_msg)
print("Normalized Root  :", preprocess_text(sample_msg))

df_clean['processed_message'] = df_clean['message'].apply(preprocess_text)
chars_b = df_clean['message'].apply(len).sum()
chars_a = df_clean['processed_message'].apply(len).sum()
words_b = df_clean['message'].apply(lambda x: len(x.split())).sum()
words_a = df_clean['processed_message'].apply(lambda x: len(x.split())).sum()

print(f"\nCorpus Reduction: Characters -{(1-chars_a/chars_b)*100:.1f}%, Words -{(1-words_a/words_b)*100:.1f}%")


## 5. Stratified 80:20 Partitioning & Scikit-learn Pipelines
We partition data using a stratified split (`random_state=42`) yielding **4,128 Training samples** and **1,032 Test samples**. To ensure zero data leakage across folds, vectorizers are bound inside Scikit-learn `Pipeline` objects.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_clean['message'],
    df_clean['label_num'],
    test_size=0.20,
    random_state=42,
    stratify=df_clean['label_num']
)

print(f"Training Fold : {len(X_train)} samples (Ham: {(y_train==0).sum()}, Spam: {(y_train==1).sum()})")
print(f"Testing Fold  : {len(X_test)} samples (Ham: {(y_test==0).sum()}, Spam: {(y_test==1).sum()})")

class TextPreprocessorTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return [preprocess_text(t) for t in X]

def build_pipeline(feature_tech: str, algorithm_name: str, max_features: int = 4000):
    is_bernoulli = (algorithm_name == "Bernoulli Naive Bayes")
    if feature_tech == "tfidf":
        vec = TfidfVectorizer(max_features=max_features, binary=is_bernoulli, sublinear_tf=(not is_bernoulli), ngram_range=(1, 2))
    else:
        vec = CountVectorizer(max_features=max_features, binary=is_bernoulli, ngram_range=(1, 2))

    if algorithm_name == "Multinomial Naive Bayes":
        clf = MultinomialNB()
    elif algorithm_name == "Bernoulli Naive Bayes":
        clf = BernoulliNB(binarize=0.0)
    elif algorithm_name == "Logistic Regression":
        clf = LogisticRegression(max_iter=1000, random_state=42)
    elif algorithm_name == "Linear SVM":
        clf = SVC(kernel='linear', probability=True, random_state=42)
    else:
        raise ValueError(f"Unknown algorithm: {algorithm_name}")

    return Pipeline([
        ('preprocessor', TextPreprocessorTransformer()),
        ('vectorizer', vec),
        ('classifier', clf)
    ])


## 6. Training Cross-Validation & Scientific Model Selection
**Rigorous Model Selection Protocol:**
- Hyperparameter tuning and model selection are conducted strictly on `X_train` using **Stratified 5-Fold Cross Validation**.
- Selection criterion: **Peak Mean CV $F_1$-score** (positive class = spam).
- The untouched test partition ($N=1,032$) is strictly isolated during this selection phase to eliminate data snooping.


In [ ]:
param_grids = {
    "Multinomial Naive Bayes": {
        'vectorizer__max_features': [3000, 4000],
        'classifier__alpha': [0.05, 0.1, 0.5, 1.0]
    },
    "Bernoulli Naive Bayes": {
        'vectorizer__max_features': [3000, 4000],
        'classifier__alpha': [0.01, 0.05, 0.1, 0.5, 1.0]
    },
    "Logistic Regression": {
        'vectorizer__max_features': [3000, 4000],
        'classifier__C': [0.5, 1.0, 5.0, 10.0],
        'classifier__solver': ['lbfgs', 'liblinear']
    },
    "Linear SVM": {
        'vectorizer__max_features': [3000, 4000],
        'classifier__C': [0.5, 1.0, 2.0]
    }
}

feature_configs = [("TF-IDF", "tfidf"), ("CountVectorizer", "count")]
algorithms = ["Multinomial Naive Bayes", "Bernoulli Naive Bayes", "Logistic Regression", "Linear SVM"]

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
trained_pipelines = {}
cv_records = []

for feat_display, feat_code in feature_configs:
    for algo in algorithms:
        exp_name = f"{feat_display} + {algo}"
        pipe = build_pipeline(feat_code, algo)
        grid = GridSearchCV(pipe, param_grids[algo], cv=cv_strategy, scoring='f1', n_jobs=-1, refit=True)
        grid.fit(X_train, y_train)
        
        best_idx = grid.best_index_
        mean_cv_f1 = float(grid.best_score_)
        std_cv_f1 = float(grid.cv_results_['std_test_score'][best_idx])
        
        trained_pipelines[exp_name] = grid.best_estimator_
        cv_records.append({
            "Experiment": exp_name,
            "Feature_Extraction": feat_display,
            "Algorithm": algo,
            "Mean_CV_F1 (%)": round(mean_cv_f1 * 100, 2),
            "Std_CV_F1 (%)": round(std_cv_f1 * 100, 2),
            "Best_Params": grid.best_params_
        })

df_cv = pd.DataFrame(cv_records)
print("=== 5-FOLD CROSS-VALIDATION TRAINING LEADERBOARD ===")
display(df_cv[["Experiment", "Mean_CV_F1 (%)", "Std_CV_F1 (%)", "Best_Params"]])

# Validation Model Selection
champion_name = df_cv.loc[df_cv["Mean_CV_F1 (%)"].idxmax(), "Experiment"]
print(f"\n★ SELECTED CHAMPION MODEL: {champion_name}")
print(f"  Selected via training 5-fold CV F1 score (Zero test set leakage).")


## 7. Comparative Test-Set Evaluation (Untouched $N = 1,032$)
With model selection finalized, we evaluate all 8 pipelines on the test partition to measure generalization.


In [ ]:
test_records = []
cms = {}

for name, pipe in trained_pipelines.items():
    y_pred = pipe.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    
    tech = "TF-IDF" if "TF-IDF" in name else "CountVectorizer"
    algo = name.replace("TF-IDF + ", "").replace("CountVectorizer + ", "")
    
    test_records.append({
        "Experiment": name,
        "Is_Validation_Champion": (name == champion_name),
        "Feature_Extraction": tech,
        "Algorithm": algo,
        "Accuracy (%)": round(acc * 100, 2),
        "Precision (%)": round(prec * 100, 2),
        "Recall (%)": round(rec * 100, 2),
        "F1-Score (%)": round(f1 * 100, 2),
        "FP (False Alarm)": cm[0, 1],
        "FN (Missed Spam)": cm[1, 0]
    })
    cms[name] = cm

df_test = pd.DataFrame(test_records)
print("\n=================== FINAL TEST SET EVALUATION TABLE ===================")
display(df_test[["Experiment", "Is_Validation_Champion", "Accuracy (%)", "Precision (%)", "Recall (%)", "F1-Score (%)", "FP (False Alarm)", "FN (Missed Spam)"]])
print("=======================================================================")


## 8. Comparative Visualizations
We visualize F1-scores, feature extraction technique comparisons, and the 8-panel confusion matrix grid.


In [ ]:
# 1. F1-Score Comparison Bar Chart
plt.figure(figsize=(12, 5))
palette = {"TF-IDF": "#2b5c8f", "CountVectorizer": "#d9534f"}
ax = sns.barplot(data=df_test, x="Algorithm", y="F1-Score (%)", hue="Feature_Extraction", palette=palette)
plt.title("Spam F1-Score Comparison: TF-IDF vs. CountVectorizer across 4 Classifiers", fontweight='bold', pad=12)
plt.ylim(90, 97)
plt.legend(title="Feature Extraction", loc="lower right")
for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(f"{h:.2f}%", (p.get_x() + p.get_width() / 2., h + 0.1), ha='center', va='bottom', fontsize=8.5, fontweight='semibold')
plt.tight_layout()
plt.show()

# 2. 8-Panel Confusion Matrix Grid
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
axes = axes.flatten()

for idx, (exp_name, cm) in enumerate(cms.items()):
    ax = axes[idx]
    total = cm.sum()
    pct = (cm / total) * 100
    labels = [
        [f"True Ham (TN)\n{cm[0,0]}\n({pct[0,0]:.1f}%)", f"False Spam (FP)\n{cm[0,1]}\n({pct[0,1]:.1f}%)"],
        [f"Missed Spam (FN)\n{cm[1,0]}\n({pct[1,0]:.1f}%)", f"True Spam (TP)\n{cm[1,1]}\n({pct[1,1]:.1f}%)"]
    ]
    cmap = "Blues" if "TF-IDF" in exp_name else "Reds"
    sns.heatmap(cm, annot=labels, fmt="", cmap=cmap, cbar=False, ax=ax, annot_kws={"fontsize": 10, "fontweight": "semibold"})
    row = df_test[df_test["Experiment"] == exp_name].iloc[0]
    champ_tag = " [★ CHAMPION]" if row["Is_Validation_Champion"] else ""
    ax.set_title(f"{exp_name}{champ_tag}\n(Acc: {row['Accuracy (%)']}% | F1: {row['F1-Score (%)']}%)", fontweight='bold')
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticklabels(["Ham", "Spam"])
    ax.set_yticklabels(["Ham", "Spam"])

plt.tight_layout()
plt.show()


## 9. Academic Benchmark Comparison with IEEE TENCON 2023 Study

### Reference Paper Metadata:
* **Title:** *A Machine Learning Approach for Efficient Spam Detection in Short Messaging System (SMS)*
* **Authors:** Robert G. de Luna et al., Polytechnic University of the Philippines
* **Conference:** 2023 IEEE Region 10 Conference (TENCON), Chiang Mai, Thailand

### Published Benchmarks (Table VII in de Luna et al.):
* **Bernoulli Naive Bayes (BNB):** Accuracy = **96.6333%**, Precision = 98.4431%, Recall = 95.5174%, $F_1$ = 92.7835%
* **Random Forest Classifier (RFC):** Accuracy = **96.0297%**, Precision = 98.5228%, Recall = 94.8699%, $F_1$ = 91.1408% *(Correct attribution: 96.03% belongs to Random Forest, not MNB)*
* **Logistic Regression (LR):** Accuracy = **95.8709%**, Precision = 98.7711%, Recall = 93.7399%, $F_1$ = 89.2555%
* **Extra Trees Classifier (ETC):** Accuracy = **95.8070%**, Precision = 98.4885%, Recall = 94.3495%, $F_1$ = 90.5663%

### Methodological Comparison & Variance Analysis:
1. **Sampling Bias & Class Balancing:** The reference paper forced an artificial **60:40 Ham:Spam partition** by undersampling ham messages. Our study preserved the natural **87.56:12.44 real-world class distribution**. Undersampling inflates apparent spam recall because spam messages appear at 3x their natural prevalence.
2. **Preprocessing Modernization:** Preserving negation markers and retaining atomic semantic tokens (`urltoken`, `currencytoken`) prevents critical signal erasure and reduces false positives (1/904 vs >10).
3. **Controlled Equivalence:** This comparison functions as an external benchmark validation rather than a direct ablation due to differences in corpora and sampling methodologies.


## 10. Interactive Real-Time Prediction Demonstration


In [ ]:
champion_pipeline = trained_pipelines[champion_name]

demo_messages = [
    "WINNER!! As a valued network customer you have been selected to receive a £900 prize reward! Call 09061701461 to claim.",
    "Hey mate, are you coming over to the library to review the Research Methodology report?",
    "URGENT! Your mobile account won 10,000 cash. Reply CLAIM to 88088 now."
]

print(f"=== REAL-TIME SMS CLASSIFICATION TEST (Model: {champion_name}) ===")
for msg in demo_messages:
    pred = champion_pipeline.predict([msg])[0]
    prob = champion_pipeline.predict_proba([msg])[0][1] * 100
    label = "🚨 SPAM" if pred == 1 else "✅ HAM (Legitimate)"
    print(f"\nMessage : \"{msg}\"")
    print(f"Verdict : {label} (Spam Probability: {prob:.1f}%)")
